# EDA — Ingeniería de datos (EPA)

**TFG:** Machine Learning explicable para analizar el estado laboral y la calidad del empleo en España.

Este notebook documenta el **análisis exploratorio** previo al modelado:

1. Volúmenes a lo largo del pipeline  
2. Cobertura temporal  
3. Targets de Fase 1 (macro) y Fase 2 (subempleo)  
4. Perfiles demográficos y laborales  
5. Nulos estructurales (no son errores)

Las figuras se exportan a `reports/memoria/eda/` para pegarlas en la memoria.

## 1. Generar / regenerar el EDA

El script `scripts/generar_eda.py` concentra el estilo y el guardado de PNG.  
Ejecuta la celda siguiente para (re)crear todas las figuras y el resumen.

**Importante:** vuelve a ejecutar esa celda (y luego la galería) después de cambiar colores o el script. El notebook hace `importlib.reload` para no quedarse con una versión antigua en memoria.


In [ ]:
import importlib
import sys
from pathlib import Path

# Raíz del repo (notebook en notebooks/ingenieria_datos/)
ROOT = Path.cwd().resolve()
if ROOT.name == "ingenieria_datos":
    ROOT = ROOT.parents[1]
elif ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import scripts.generar_eda as generar_eda

# Recarga el script siempre (si no, Jupyter guarda la versión antigua en memoria)
generar_eda = importlib.reload(generar_eda)

EDA_DIR = generar_eda.EDA_DIR
RESUMEN_JSON = generar_eda.RESUMEN_JSON
RESUMEN_MD = generar_eda.RESUMEN_MD

resumen = generar_eda.generar_eda_completo()
resumen


## 2. Galería de figuras

Vista rápida de lo generado (también están en disco para la memoria).

In [ ]:
from IPython.display import Image, Markdown, display

figuras = sorted(EDA_DIR.glob("*.png"))
print(f"{len(figuras)} figuras en {EDA_DIR.relative_to(ROOT)}")
for fig in figuras:
    display(Markdown(f"### `{fig.name}`"))
    # Leer bytes (no filename): evita caché del visor del notebook
    display(Image(data=fig.read_bytes(), width=720))


## 3. Resumen numérico (para citar en la memoria)

In [ ]:
import json

with open(RESUMEN_JSON, encoding="utf-8") as f:
    datos = json.load(f)

print("=== VOLÚMENES ===")
print(f"Interim:     {datos['interim_filas']:>12,}  ({datos['interim_columnas']} cols)")
print(f"Fase 1 16-64:{datos['fase1_filas']:>12,}  ({datos['fase1_columnas']} cols)")
print(f"Fase 2 ocup.:{datos['fase2_filas']:>12,}  ({datos['fase2_columnas']} cols)")
print()
print("=== TARGET FASE 1 ===")
print(f"Ocupado  {datos['fase1_pct_ocupado']:5.2f}%")
print(f"Parado   {datos['fase1_pct_parado']:5.2f}%")
print(f"Inactivo {datos['fase1_pct_inactivo']:5.2f}%")
print()
print("=== TARGET FASE 2 ===")
print(f"Tasa subempleo (AOI=03): {datos['fase2_tasa_subempleo']:.2f}%")
print()
print("Markdown:", RESUMEN_MD.relative_to(ROOT))

## 4. Decisiones que este EDA ilustra

| Decisión | Evidencia en el EDA |
|---|---|
| Filtrar **16–64** en Fase 1 | Volumen Fase 1 < interim; evita dominancia de jubilación |
| Fase 2 = solo ocupados | Target binario con ~7–8% positivos |
| Horas / jornada en Fase 2 | Boxplots y barras: separación clara subempleo vs no |
| Nulos altos en origen/contrato | Figura de nulos (azul/ámbar = dataset, no estado laboral) |
| Educación ↔ estado laboral | Stacked bar: más educación, más ocupación |
| Estabilidad temporal | Evolución trimestral estado laboral y tasa de subempleo |
| Lectura territorial | Top CCAA por % parado (muestra EPA) |

**Colores:** verde petróleo / naranja / gris = estados laborales; azul / ámbar = datasets Fase 1 / Fase 2 (no mezclar en la memoria).

Con esto la **Fase 0 (ingeniería de datos)** queda documentada visualmente para la memoria.